# Unique3D on Kaggle GPU — self-hosted `FULL_AVATAR_3D_PROVIDER=unique3d` backend

Runs the official [Unique3D](https://github.com/AiuniAI/Unique3D) Gradio demo on Kaggle's free
weekly 30 GPU-hour quota (P100 / T4x2), then tunnels it out with ngrok.

**Why this instead of a custom endpoint:** the project's `avatar3d_providers.Unique3DAvatarProvider`
already knows how to call *any* Gradio app that exposes the `/generate3dv2` API (that's how it talks
to the public `Wuvin/Unique3D` community Space today). `gradio_client.Client(...)` accepts a raw URL,
not just a Hugging Face Space id — so pointing it at your own Kaggle-hosted instance needs zero code
changes in the main repo, and you skip the shared/rate-limited ZeroGPU queue entirely.

**Before running:**
1. Kaggle notebook settings -> Accelerator -> GPU (T4 x2 or P100). Settings -> Internet -> On.
2. Add your ngrok authtoken as a Kaggle Secret named `NGROK_AUTHTOKEN`
   (free account at https://dashboard.ngrok.com/get-started/your-authtoken).
3. Run all cells. The last cell prints the public URL and blocks to keep the tunnel alive —
   stop it manually when you're done (Kaggle sessions cap at 9h/run, 30h/week).

**On your main project**, once the URL is printed, set:
```
FULL_AVATAR_3D_PROVIDER=unique3d
UNIQUE3D_HF_SPACE=<the ngrok URL printed below>
```
(leave `UNIQUE3D_HF_TOKEN` unset — that's only needed for the public HF Space's ZeroGPU auth,
not for your own instance).

In [ ]:
!nvidia-smi

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/AiuniAI/Unique3D.git
%cd /kaggle/working/Unique3D

In [ ]:
# Kaggle kernels already ship a CUDA-enabled torch; install on top of it rather than
# creating a conda env (Kaggle's default kernel is a single shared environment).
!pip install -q ninja
!pip install -q diffusers==0.27.2
!pip install -q mmcv-full -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.3.1/index.html
!pip install -q -r requirements.txt
!pip install -q pyngrok huggingface_hub

In [ ]:
# Checkpoints aren't published as a plain model repo — pull them from the public
# Space's own repo (repo_type="space") into this checkout's ckpt/ dir.
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Wuvin/Unique3D",
    repo_type="space",
    allow_patterns=["ckpt/*"],
    local_dir="/kaggle/working/Unique3D",
)
!ls ckpt

In [ ]:
import subprocess, sys, time

server = subprocess.Popen(
    [sys.executable, "app/gradio_local.py", "--port", "7860"],
    cwd="/kaggle/working/Unique3D",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

# Wait for the demo to actually bind the port before tunneling.
import socket
for _ in range(180):
    if server.poll() is not None:
        print(server.stdout.read())
        raise RuntimeError("gradio_local.py exited early — see log above")
    try:
        with socket.create_connection(("127.0.0.1", 7860), timeout=1):
            break
    except OSError:
        time.sleep(2)
else:
    raise RuntimeError("Timed out waiting for port 7860 to come up")

print("Unique3D gradio demo is up on port 7860.")

In [ ]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok, conf

authtoken = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
conf.get_default().auth_token = authtoken

tunnel = ngrok.connect(7860, "http")
print(f"\nPublic URL: {tunnel.public_url}\n")
print("Set on your main project:")
print("  FULL_AVATAR_3D_PROVIDER=unique3d")
print(f"  UNIQUE3D_HF_SPACE={tunnel.public_url}")

In [ ]:
# Keep the kernel (and tunnel) alive. Interrupt/stop this cell to end the session
# — the ngrok tunnel and gradio server both die with it.
import time
try:
    while True:
        line = server.stdout.readline()
        if line:
            print(line, end="")
        elif server.poll() is not None:
            print("gradio_local.py exited")
            break
        else:
            time.sleep(1)
except KeyboardInterrupt:
    print("stopped")